
# Adaptive SPSA-calibrated GRAPE

This notebook tests the next idea after the feature-RBF experiment.

The RBF residual often became either inactive or poorly calibrated. The good pulses seemed to come mostly from a measured search loop: GRAPE proposed a pulse, SPSA-like perturbations explored nearby pulses, and binary measurements selected the best one.

Here we remove the RBF correction and instead learn a richer physical parameter vector for the simulator:

```text
measured pulses -> fit physical parameters -> AD-GRAPE on calibrated simulator -> SPSA candidate cloud -> binary measurements -> repeat
```

Main differences from the RBF notebook:

1. The learned model is still a physical Hamiltonian/Lindblad model, not an arbitrary residual function.
2. GRAPE starts each round from the best measured pulse.
3. Training data is generated by SPSA-style paired perturbations around the GRAPE proposal, not just random local noise.
4. The first few rounds use unitary GRAPE for speed and robustness; later rounds switch to non-unitary GRAPE using the calibrated T1/T2 parameters.


In [ ]:

from dataclasses import replace

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from hybrid_residual_grape import (
    FockPhysicsModel,
    HybridGrapeConfig,
    MeasurementResponse,
    PhysicalCalibrationConfig,
    PhysicsParams,
    RBFResidualConfig,
    SimulationConfig,
    adaptive_shot_counts,
    append_dataset,
    beta_normal_lower_bound,
    calibration_parameter_summary,
    empty_rbf_model,
    fit_physical_parameters,
    observed_probability_from_physical,
    optimize_hybrid_grape,
    params_from_calibration_raw,
    physical_parameter_names,
    physical_parameter_size,
    sample_binomial_measurements,
)
from hybrid_residual_grape.config import (
    khz_to_rad_per_us,
    qubit_t1_us,
    qubit_t2_us,
    storage_t1_us,
    storage_t2_us,
)

jax.config.update("jax_enable_x64", True)



## Setup

`calibration_model` is the model family used for fitting physical parameters. `true_model` is only the hidden simulator used here to mimic the experiment.

The hidden model includes realistic small errors: chi and detuning offsets of a few kHz, small cavity self-Kerr, drive-scale and phase errors, plus shortened T1/T2. The optimizer only sees binomial measurement counts.


In [ ]:

seed = 5050
key = jax.random.key(seed)

q = SimulationConfig(
    n_cav=25,
    target_n=2,
    initial_cavity_n=0,
    initial_qubit_state=0,
    t_drive=1.408,
    ndt_drive=80,
    num_coeffs=20,
    spline_degree=2,
    spline_skip_left=2,
    spline_skip_right=2,
    param_clip=2.0,
)

# Keep the OPX coefficient bound at [-2, 2], but use the stronger calibrated
# drive conversion that worked better in the previous notebook.
drive_coupling = 40.0
config_params = replace(PhysicsParams(), mu_qub=drive_coupling, mu_cav=drive_coupling)

# Starting model: deliberately imperfect. No self-Kerr, no detuning, no drive phase,
# and no decoherence in the unitary model.
nominal_params = replace(
    config_params,
    cavity_self_kerr=0.0,
    cavity_detuning=0.0,
    qubit_detuning=0.0,
    cavity_phase=0.0,
    qubit_phase=0.0,
    qubit_t1_us=None,
    qubit_t2_us=None,
    cavity_t1_us=None,
    cavity_t2_us=None,
)
unitary_nominal_model = FockPhysicsModel(q, nominal_params)

# Reference lifetimes are known from configuration.json and used as the center of
# the fitted non-unitary model.
reference_params = replace(
    nominal_params,
    qubit_t1_us=qubit_t1_us(),
    qubit_t2_us=qubit_t2_us(),
    cavity_t1_us=storage_t1_us(),
    cavity_t2_us=storage_t2_us(),
)
calibration_model = FockPhysicsModel(q, reference_params)

# Hidden true experiment. These offsets are intentionally small enough to be realistic,
# but large enough that calibration should matter.
lifetime_scale = 0.80
true_params = replace(
    config_params,
    chi=config_params.chi + khz_to_rad_per_us(3.0),
    cavity_self_kerr=config_params.cavity_self_kerr + khz_to_rad_per_us(0.12),
    cavity_detuning=khz_to_rad_per_us(5.0),
    qubit_detuning=khz_to_rad_per_us(-4.0),
    mu_qub=config_params.mu_qub * 1.012,
    mu_cav=config_params.mu_cav * 0.988,
    cavity_phase=0.025,
    qubit_phase=-0.020,
    qubit_t1_us=lifetime_scale * qubit_t1_us(),
    qubit_t2_us=lifetime_scale * qubit_t2_us(),
    cavity_t1_us=lifetime_scale * storage_t1_us(),
    cavity_t2_us=lifetime_scale * storage_t2_us(),
)
true_model = FockPhysicsModel(q, true_params)

# Measurement channel for the long selective pi pulse plus qubit readout.
# The optimizer only sees Bernoulli samples from this observed probability,
# while the plots can still show the hidden physical Fock probability.
measurement_response = MeasurementResponse(
    false_positive=0.010,
    true_positive=0.900,
    curvature=0.10,
)

response_test = jnp.linspace(0.0, 1.0, 6)
print("measurement response P_phys -> Pr(yes):")
for p_phys, p_obs in zip(
    response_test,
    observed_probability_from_physical(response_test, measurement_response),
):
    print(f"  {float(p_phys):.2f} -> {float(p_obs):.3f}")

print("parameter size:", unitary_nominal_model.parameter_size)
print("calibrated parameter names:", physical_parameter_names())
print("calibrated parameter size:", physical_parameter_size())
print("unitary nominal collapse operators:", len(unitary_nominal_model.collapse_ops))
print("calibration/reference collapse operators:", len(calibration_model.collapse_ops))
print("true experiment collapse operators:", len(true_model.collapse_ops))
print("basis endpoint max:", float(jnp.max(jnp.abs(unitary_nominal_model.bsplines_edges[:, [0, -1]]))))
print("max active B-splines:", int(jnp.max(jnp.sum(unitary_nominal_model.bsplines_mids > 1e-12, axis=0))))



## Hyperparameters

The loop starts with one optional long unitary GRAPE pre-optimization. This is cheap compared with density-matrix GRAPE and gives the closed-loop routine a sensible first anchor before spending binary measurements.

After that, the closed-loop loop has two possible phases:

- `unitary_grape_rounds`: optional extra unitary calibrated rounds inside the loop. By default this is `0` because the long unitary pre-GRAPE already did that job.
- later rounds: use the non-unitary calibrated model for GRAPE, including fitted T1/T2.

The SPSA radius decays slowly. The idea is to explore coarsely while the pulse is bad, then test smaller perturbations once the measured fidelity is high.

Open-system calibration is expensive because it differentiates through density matrices. The notebook therefore fits calibration parameters on small fixed-size mini-batches (`calibration_batch_size`) instead of compiling one huge full-dataset Adam loop.


In [ ]:

empty_residual = empty_rbf_model(unitary_nominal_model.parameter_size, RBFResidualConfig(max_centers=1))

unitary_grape_config = HybridGrapeConfig(
    maxiter=220,
    memory_size=18,
    noise_samples=2,
    control_noise_std=0.006,
    residual_support_penalty=0.0,
    residual_size_penalty=0.0,
    amplitude_l2=4e-5,
    smoothness_l2=6e-5,
    param_clip=q.param_clip,
    grad_clip_norm=30.0,
)

run_initial_unitary_pregrape = True
initial_unitary_grape_steps = 1000
initial_unitary_pregrape_config = replace(
    unitary_grape_config,
    maxiter=initial_unitary_grape_steps,
    memory_size=25,
    noise_samples=1,
    control_noise_std=0.0,
)

nonunitary_grape_config = HybridGrapeConfig(
    maxiter=70,
    memory_size=12,
    noise_samples=1,
    control_noise_std=0.0,
    residual_support_penalty=0.0,
    residual_size_penalty=0.0,
    amplitude_l2=4e-5,
    smoothness_l2=6e-5,
    param_clip=q.param_clip,
    grad_clip_norm=30.0,
)

calibration_config = PhysicalCalibrationConfig(
    fit_decoherence=True,
    fit_steps=80,
    batch_size=8,
    learning_rate=0.030,
    prior_strength=1e-3,
    max_chi_offset_khz=8.0,
    max_cavity_detuning_khz=10.0,
    max_qubit_detuning_khz=10.0,
    max_cavity_self_kerr_abs_khz=1.0,
    max_drive_scale_fraction=0.08,
    max_cavity_phase_rad=0.12,
    max_qubit_phase_rad=0.12,
    max_lifetime_log_scale=0.80,
    measurement_response=measurement_response,
)

rounds = 80
unitary_grape_rounds = 0
min_fit_points = 8
calibration_max_points = 260
calibration_batch_size = calibration_config.batch_size

spsa_pairs_center = 8
spsa_pairs_anchor = 4
line_candidate_count = 4
spsa_step_initial = 0.075
spsa_step_final = 0.018
anchor_spsa_fraction = 0.55
min_candidate_distance = 1e-3

low_shots = 800
medium_shots = 3000
high_shots = 10000
medium_threshold = 0.86
high_threshold = 0.94
center_min_shots = 4000
anchor_min_shots = 4000
selection_z = 1.5
shot_time_us = 400.0

max_candidate_pool_size = 2 + line_candidate_count + 2 * spsa_pairs_center + 2 * spsa_pairs_anchor
print("initial unitary pre-GRAPE steps:", initial_unitary_grape_steps if run_initial_unitary_pregrape else 0)
print("closed-loop unitary GRAPE rounds:", unitary_grape_rounds)
print("max candidate pool size:", max_candidate_pool_size)
print("calibration mini-batch size:", calibration_batch_size)
print("max round shots:", max_candidate_pool_size * high_shots)
print("max measurement time per round [s]:", max_candidate_pool_size * high_shots * shot_time_us / 1e6)



## Experiment hook

In the notebook this is a hidden JAX simulator plus binomial shot sampling. On hardware this should become the OPX call returning successes and shots for each pulse.


In [ ]:

def measure_on_experiment(controls, key, shots):
    return sample_binomial_measurements(
        true_model,
        controls,
        key,
        shots=shots,
        response=measurement_response,
    )



## Helper functions

The important helper is `make_spsa_candidate_pool`. It creates paired plus/minus perturbations around the pulse just optimized by GRAPE, plus a smaller set around the current measured champion. All candidates are clipped to the hardware coefficient range.


In [ ]:

def without_decoherence(params):
    return replace(
        params,
        qubit_t1_us=None,
        qubit_t2_us=None,
        cavity_t1_us=None,
        cavity_t2_us=None,
    )


def clip_controls(controls):
    return jnp.clip(controls, -q.param_clip, q.param_clip)


def deduplicate_controls(pool_controls, min_distance):
    kept = []
    for candidate in pool_controls:
        if not kept:
            kept.append(candidate)
            continue
        distances = jnp.array([jnp.linalg.norm(candidate - previous) for previous in kept])
        if float(jnp.min(distances)) > min_distance:
            kept.append(candidate)
    return jnp.stack(kept)


def spsa_radius(round_index):
    if rounds <= 1:
        return spsa_step_final
    frac = round_index / (rounds - 1)
    return float(spsa_step_initial * (spsa_step_final / spsa_step_initial) ** frac)


def make_spsa_candidate_pool(anchor_controls, center_controls, key, radius):
    key, center_key, anchor_key = jax.random.split(key, 3)
    anchor_controls = jnp.asarray(anchor_controls)
    center_controls = jnp.asarray(center_controls)

    center_dirs = jax.random.rademacher(
        center_key,
        (spsa_pairs_center, center_controls.shape[0]),
        dtype=center_controls.dtype,
    )
    center_pairs = jnp.concatenate(
        [
            center_controls[None, :] + radius * center_dirs,
            center_controls[None, :] - radius * center_dirs,
        ],
        axis=0,
    )

    anchor_dirs = jax.random.rademacher(
        anchor_key,
        (spsa_pairs_anchor, anchor_controls.shape[0]),
        dtype=anchor_controls.dtype,
    )
    anchor_radius = anchor_spsa_fraction * radius
    anchor_pairs = jnp.concatenate(
        [
            anchor_controls[None, :] + anchor_radius * anchor_dirs,
            anchor_controls[None, :] - anchor_radius * anchor_dirs,
        ],
        axis=0,
    )

    line_fracs = jnp.linspace(0.2, 0.8, line_candidate_count)[:, None]
    line = anchor_controls[None, :] + line_fracs * (center_controls - anchor_controls)[None, :]

    pool = jnp.concatenate(
        [
            anchor_controls[None, :],
            center_controls[None, :],
            line,
            center_pairs,
            anchor_pairs,
        ],
        axis=0,
    )
    pool = deduplicate_controls(clip_controls(pool), min_candidate_distance)
    return pool, key


def select_calibration_data(controls, successes, shots, max_points):
    if controls.shape[0] <= max_points:
        return controls, successes, shots
    return controls[-max_points:], successes[-max_points:], shots[-max_points:]


def log_infidelity(probability):
    return -jnp.log10(jnp.maximum(1.0 - probability, 1e-8))


def append_measurements(controls, successes, shots, model_probability):
    global dataset_controls, dataset_successes, dataset_shots, dataset_model_probability
    dataset_controls, dataset_successes, dataset_shots, dataset_model_probability = append_dataset(
        dataset_controls,
        dataset_successes,
        dataset_shots,
        dataset_model_probability,
        controls,
        successes,
        shots,
        model_probability,
    )
    return int(jnp.sum(shots))



## Closed-loop SPSA-calibrated GRAPE

Before the loop, the notebook optionally runs one long unitary GRAPE optimization from a small random pulse. This produces the first measured anchor.

Each closed-loop round then:

1. Fit physical parameters from recent measured pulses using mini-batches.
2. Run GRAPE from the best measured pulse.
3. Generate SPSA paired perturbations around the GRAPE pulse.
4. Measure all candidates with adaptive shot counts.
5. Add everything to the calibration dataset.
6. Promote the candidate with the best lower-confidence score.

The printed `mode` is `unitary` only if `unitary_grape_rounds > 0`; otherwise the measured loop goes straight to `open` GRAPE after the one-time unitary pre-GRAPE.


In [ ]:

key, init_key = jax.random.split(key)
initial_seed_controls = 0.08 * jax.random.normal(init_key, (unitary_nominal_model.parameter_size,))
initial_seed_controls = clip_controls(initial_seed_controls)

if run_initial_unitary_pregrape:
    key, pregrape_key = jax.random.split(key)
    initial_controls, initial_unitary_history, initial_unitary_summary, key = optimize_hybrid_grape(
        unitary_nominal_model,
        empty_residual,
        initial_seed_controls,
        pregrape_key,
        initial_unitary_pregrape_config,
    )
    initial_unitary_pred = float(unitary_nominal_model.photon_probability(initial_controls))
    initial_unitary_true = float(true_model.photon_probability(initial_controls))
    print("initial unitary pre-GRAPE nominal P_n:", initial_unitary_pred)
    print("initial unitary pre-GRAPE hidden true P_n:", initial_unitary_true)
else:
    initial_controls = initial_seed_controls
    initial_unitary_history = None
    initial_unitary_summary = None
    print("initial unitary pre-GRAPE skipped")

key, init_measure_key = jax.random.split(key)

calibration_raw = jnp.zeros((physical_parameter_size(calibration_config),), dtype=jnp.float64)
calibrated_params = params_from_calibration_raw(
    calibration_raw,
    nominal_params,
    reference_params,
    calibration_config,
)

init_successes, init_shots, init_true_probability, key = measure_on_experiment(
    initial_controls[None, :],
    init_measure_key,
    shots=medium_shots,
)
init_measured = init_successes / init_shots
init_lcb = beta_normal_lower_bound(init_successes, init_shots, z=selection_z)

best_controls = initial_controls
best_successes = init_successes[0]
best_shots = init_shots[0]
best_measured = float(init_measured[0])
best_lcb = float(init_lcb[0])
best_true_diagnostic = float(init_true_probability[0])

# Dataset for parameter calibration.
dataset_controls = None
dataset_successes = None
dataset_shots = None
dataset_model_probability = None
initial_model_probability = observed_probability_from_physical(
    unitary_nominal_model.population_probability(initial_controls[None, :]),
    measurement_response,
)
total_measurements = append_measurements(
    initial_controls[None, :],
    init_successes,
    init_shots,
    initial_model_probability,
)

progress_rows = []
param_history = []
calibration_histories = []
grape_histories = []

print(
    f"initial measured={best_measured:.4f} lcb={best_lcb:.4f} "
    f"true_diagnostic={best_true_diagnostic:.4f} shots={int(best_shots)}"
)

pbar = tqdm(range(rounds), desc="SPSA-calibrated GRAPE")

for round_index in pbar:
    if dataset_controls is not None and dataset_controls.shape[0] >= min_fit_points:
        fit_controls, fit_successes, fit_shots = select_calibration_data(
            dataset_controls,
            dataset_successes,
            dataset_shots,
            calibration_max_points,
        )
        calibration_raw, calibrated_params, fit_history = fit_physical_parameters(
            calibration_model,
            nominal_params,
            reference_params,
            fit_controls,
            fit_successes,
            fit_shots,
            initial_raw=calibration_raw,
            config=calibration_config,
        )
        calibration_histories.append(fit_history)
        fit_loss = float(fit_history[-1, 0])
        fit_nll = float(fit_history[-1, 1])
    else:
        fit_loss = float("nan")
        fit_nll = float("nan")

    param_summary = calibration_parameter_summary(
        calibration_raw,
        calibrated_params,
        nominal_params,
        reference_params,
        calibration_config,
    )
    param_history.append(param_summary)

    use_open_system_grape = round_index >= unitary_grape_rounds
    grape_params = calibrated_params if use_open_system_grape else without_decoherence(calibrated_params)
    grape_model = FockPhysicsModel(q, grape_params)
    grape_config = nonunitary_grape_config if use_open_system_grape else unitary_grape_config
    mode_value = 1.0 if use_open_system_grape else 0.0
    mode_name = "open" if use_open_system_grape else "unitary"

    proposed_controls, grape_history, grape_summary, key = optimize_hybrid_grape(
        grape_model,
        empty_residual,
        best_controls,
        key,
        grape_config,
    )
    grape_histories.append(grape_history)

    radius = spsa_radius(round_index)
    candidate_controls, key = make_spsa_candidate_pool(best_controls, proposed_controls, key, radius)
    physical_predicted_probability = grape_model.population_probability(candidate_controls)
    predicted_probability = observed_probability_from_physical(
        physical_predicted_probability,
        measurement_response,
    )
    shot_counts = adaptive_shot_counts(
        predicted_probability,
        low_shots=low_shots,
        medium_shots=medium_shots,
        high_shots=high_shots,
        medium_threshold=medium_threshold,
        high_threshold=high_threshold,
    )

    anchor_idx = 0
    center_idx = int(jnp.argmin(jnp.linalg.norm(candidate_controls - proposed_controls[None, :], axis=1)))
    shot_counts = shot_counts.at[anchor_idx].set(jnp.maximum(shot_counts[anchor_idx], anchor_min_shots))
    shot_counts = shot_counts.at[center_idx].set(jnp.maximum(shot_counts[center_idx], center_min_shots))

    successes, measured_shots, true_probability, key = measure_on_experiment(
        candidate_controls,
        key,
        shots=shot_counts,
    )
    total_measurements += append_measurements(
        candidate_controls,
        successes,
        measured_shots,
        predicted_probability,
    )

    measured_probability = successes / measured_shots
    lower_bound = beta_normal_lower_bound(successes, measured_shots, z=selection_z)
    best_in_batch = int(jnp.argmax(lower_bound))

    old_best_measured = best_measured
    old_best_lcb = best_lcb
    old_best_true = best_true_diagnostic

    accepted = bool(float(lower_bound[best_in_batch]) > best_lcb)
    if accepted:
        best_controls = candidate_controls[best_in_batch]
        best_successes = successes[best_in_batch]
        best_shots = measured_shots[best_in_batch]
        best_measured = float(measured_probability[best_in_batch])
        best_lcb = float(lower_bound[best_in_batch])
        best_true_diagnostic = float(true_probability[best_in_batch])

    center_measured = float(measured_probability[center_idx])
    center_lcb = float(lower_bound[center_idx])
    center_true = float(true_probability[center_idx])
    center_pred = float(predicted_probability[center_idx])
    round_best_true = float(jnp.max(true_probability))

    progress_rows.append(
        jnp.array(
            [
                float(total_measurements),
                best_measured,
                best_lcb,
                best_true_diagnostic,
                center_measured,
                center_lcb,
                center_true,
                center_pred,
                float(measured_probability[best_in_batch]),
                float(lower_bound[best_in_batch]),
                float(true_probability[best_in_batch]),
                fit_loss,
                fit_nll,
                float(jnp.sum(measured_shots)),
                float(dataset_controls.shape[0]),
                mode_value,
                radius,
                old_best_measured,
                old_best_lcb,
                old_best_true,
                float(predicted_probability[best_in_batch]),
            ]
        )
    )

    pbar.set_postfix(
        {
            "mode": mode_name,
            "best_lcb": f"{best_lcb:.3f}",
            "best_true": f"{best_true_diagnostic:.4f}",
            "center": f"{center_measured:.3f}",
            "radius": f"{radius:.3f}",
            "shots": int(jnp.sum(measured_shots)),
        }
    )
    tqdm.write(
        f"round {round_index:02d} mode={mode_name} "
        f"old={old_best_measured:.4f}/{old_best_lcb:.4f} "
        f"candidate={float(measured_probability[best_in_batch]):.4f}/{float(lower_bound[best_in_batch]):.4f} "
        f"accept={int(accepted)} best={best_measured:.4f}/{best_lcb:.4f} "
        f"true={best_true_diagnostic:.4f} center_pred={center_pred:.4f} "
        f"radius={radius:.3f} data={dataset_controls.shape[0]}"
    )

progress = jnp.stack(progress_rows) if progress_rows else jnp.zeros((0, 21))
print("total binary measurements:", total_measurements)
print("best measured P_n:", best_measured)
print("best lower confidence bound:", best_lcb)
print("best true diagnostic P_n:", best_true_diagnostic)
print("last calibrated parameters:")
for key_name, value in param_history[-1].items():
    print(f"  {key_name}: {value:.6g}")



## Progress plots

The first panel is the experiment-like quantity: measured success rates and lower-confidence scores. The second panel uses the hidden true probability, only available in simulation. The third panel shows whether the fitted simulator plus measurement response is predicting the observed GRAPE-center success probability well. The fourth panel shows the switch from unitary to non-unitary GRAPE.


In [ ]:

if progress.shape[0]:
    xs = progress[:, 0]
    mode = progress[:, 15]

    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True, constrained_layout=True)

    axes[0].plot(xs, progress[:, 1], label="best measured")
    axes[0].plot(xs, progress[:, 2], label="best lower-confidence score")
    axes[0].plot(xs, progress[:, 4], alpha=0.7, label="GRAPE center measured")
    axes[0].plot(xs, progress[:, 9], alpha=0.7, label="round best candidate LCB")
    axes[0].set_ylabel("observed Pr(yes)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(xs, progress[:, 3], label="accepted best true diagnostic")
    axes[1].plot(xs, progress[:, 6], alpha=0.7, label="GRAPE center true diagnostic")
    axes[1].plot(xs, progress[:, 10], alpha=0.7, label="round best candidate true diagnostic")
    axes[1].set_ylabel("hidden true P_n")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(xs, progress[:, 7], label="calibrated observed prediction for center")
    axes[2].plot(xs, progress[:, 6], label="hidden true center")
    axes[2].plot(xs, progress[:, 20], alpha=0.7, label="observed prediction for selected candidate")
    axes[2].plot(xs, progress[:, 10], alpha=0.7, label="hidden true selected candidate")
    axes[2].set_ylabel("probability")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    axes[3].plot(xs, progress[:, 16], label="SPSA radius")
    axes[3].plot(xs, progress[:, 13] / high_shots, label="round shots / high_shots")
    axes[3].step(xs, mode, where="post", label="GRAPE mode: 0 unitary, 1 open")
    axes[3].set_xlabel("total binary measurements")
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()

    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(xs, log_infidelity(progress[:, 3]), label="accepted best true")
    ax.plot(xs, log_infidelity(progress[:, 6]), label="GRAPE center true")
    ax.set_xlabel("total binary measurements")
    ax.set_ylabel("-log10(1 - P_n)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
else:
    print("Run the closed-loop cell first.")



## Parameter traces

Dashed lines are the hidden true values, available only because this is a simulation. The fitted values do not need to match exactly to be useful; what matters is whether the calibrated model predicts the measured pulse neighborhood better and leads GRAPE toward better pulses.


In [ ]:

if param_history:
    xs = jnp.arange(len(param_history))

    chi = jnp.array([row["chi_offset_khz"] for row in param_history])
    cav_det = jnp.array([row["cavity_detuning_khz"] for row in param_history])
    qub_det = jnp.array([row["qubit_detuning_khz"] for row in param_history])
    kerr = jnp.array([row["cavity_self_kerr_khz"] for row in param_history])
    mu_q = jnp.array([row["mu_qub"] for row in param_history])
    mu_c = jnp.array([row["mu_cav"] for row in param_history])
    cav_phase = jnp.array([row["cavity_phase_rad"] for row in param_history])
    qub_phase = jnp.array([row["qubit_phase_rad"] for row in param_history])
    qt1 = jnp.array([row["qubit_T1_scale"] for row in param_history])
    qt2 = jnp.array([row["qubit_T2_scale"] for row in param_history])
    ct1 = jnp.array([row["cavity_T1_scale"] for row in param_history])
    ct2 = jnp.array([row["cavity_T2_scale"] for row in param_history])

    true_chi_offset = (true_params.chi - nominal_params.chi) * 1000.0 / (2.0 * jnp.pi)
    true_kerr = true_params.cavity_self_kerr * 1000.0 / (2.0 * jnp.pi)
    true_cav_det = true_params.cavity_detuning * 1000.0 / (2.0 * jnp.pi)
    true_qub_det = true_params.qubit_detuning * 1000.0 / (2.0 * jnp.pi)

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True, constrained_layout=True)

    axes[0].plot(xs, chi, label="chi offset")
    axes[0].plot(xs, cav_det, label="cavity detuning")
    axes[0].plot(xs, qub_det, label="qubit detuning")
    axes[0].plot(xs, kerr, label="cavity self-Kerr")
    axes[0].axhline(float(true_chi_offset), color="tab:blue", linestyle="--", alpha=0.4)
    axes[0].axhline(float(true_cav_det), color="tab:orange", linestyle="--", alpha=0.4)
    axes[0].axhline(float(true_qub_det), color="tab:green", linestyle="--", alpha=0.4)
    axes[0].axhline(float(true_kerr), color="tab:red", linestyle="--", alpha=0.4)
    axes[0].set_ylabel("kHz")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(xs, mu_q / nominal_params.mu_qub, label="mu_qub scale")
    axes[1].plot(xs, mu_c / nominal_params.mu_cav, label="mu_cav scale")
    axes[1].plot(xs, cav_phase, label="cavity phase [rad]")
    axes[1].plot(xs, qub_phase, label="qubit phase [rad]")
    axes[1].axhline(float(true_params.mu_qub / nominal_params.mu_qub), color="tab:blue", linestyle="--", alpha=0.4)
    axes[1].axhline(float(true_params.mu_cav / nominal_params.mu_cav), color="tab:orange", linestyle="--", alpha=0.4)
    axes[1].axhline(float(true_params.cavity_phase), color="tab:green", linestyle="--", alpha=0.4)
    axes[1].axhline(float(true_params.qubit_phase), color="tab:red", linestyle="--", alpha=0.4)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(xs, qt1, label="qubit T1 scale")
    axes[2].plot(xs, qt2, label="qubit T2 scale")
    axes[2].plot(xs, ct1, label="cavity T1 scale")
    axes[2].plot(xs, ct2, label="cavity T2 scale")
    axes[2].axhline(lifetime_scale, color="black", linestyle="--", linewidth=1, label="hidden true scale")
    axes[2].set_xlabel("closed-loop round")
    axes[2].set_ylabel("scale vs config.json")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.show()
else:
    print("Run the closed-loop cell first.")



## Calibration diagnostic on measured pulses

This asks whether the calibrated physical model predicts the measured dataset better than the original nominal model. It is not the final goal, but it helps catch a bad calibration fit.


In [ ]:

if dataset_controls is not None and dataset_controls.shape[0] > 0:
    calibrated_model = FockPhysicsModel(q, calibrated_params)
    measured = dataset_successes / dataset_shots
    nominal_physical_pred = unitary_nominal_model.population_probability(dataset_controls)
    calibrated_physical_pred = calibrated_model.population_probability(dataset_controls)
    nominal_pred = observed_probability_from_physical(nominal_physical_pred, measurement_response)
    calibrated_pred = observed_probability_from_physical(calibrated_physical_pred, measurement_response)
    true_diag = true_model.population_probability(dataset_controls)

    nominal_measured_mae = jnp.mean(jnp.abs(measured - nominal_pred))
    calibrated_measured_mae = jnp.mean(jnp.abs(measured - calibrated_pred))
    nominal_true_mae = jnp.mean(jnp.abs(true_diag - nominal_physical_pred))
    calibrated_true_mae = jnp.mean(jnp.abs(true_diag - calibrated_physical_pred))

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), constrained_layout=True)
    axes = axes.ravel()

    axes[0].scatter(nominal_pred, measured, c=dataset_shots, cmap="viridis", alpha=0.8)
    axes[0].plot([0, 1], [0, 1], color="black", linewidth=1)
    axes[0].set_xlabel("nominal predicted Pr(yes)")
    axes[0].set_ylabel("measured successes/shots")
    axes[0].set_title("before calibration")
    axes[0].grid(True, alpha=0.3)

    sc = axes[1].scatter(calibrated_pred, measured, c=dataset_shots, cmap="viridis", alpha=0.8)
    axes[1].plot([0, 1], [0, 1], color="black", linewidth=1)
    axes[1].set_xlabel("calibrated predicted Pr(yes)")
    axes[1].set_ylabel("measured successes/shots")
    axes[1].set_title("after calibration")
    axes[1].grid(True, alpha=0.3)
    fig.colorbar(sc, ax=axes[1], label="shots")

    bins = jnp.linspace(
        0.0,
        float(jnp.maximum(jnp.max(jnp.abs(measured - nominal_pred)), jnp.max(jnp.abs(measured - calibrated_pred)))) + 1e-6,
        35,
    )
    axes[2].hist(jax.device_get(jnp.abs(measured - nominal_pred)), bins=jax.device_get(bins), alpha=0.6, label="|measured - nominal|")
    axes[2].hist(jax.device_get(jnp.abs(measured - calibrated_pred)), bins=jax.device_get(bins), alpha=0.6, label="|measured - calibrated|")
    axes[2].set_xlabel("absolute probability error")
    axes[2].set_ylabel("count")
    axes[2].set_title("measured error")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    axes[3].scatter(true_diag - nominal_physical_pred, calibrated_physical_pred - nominal_physical_pred, c=dataset_shots, cmap="plasma", alpha=0.8)
    lim = float(jnp.maximum(0.03, 1.05 * jnp.max(jnp.abs(jnp.concatenate([true_diag - nominal_physical_pred, calibrated_physical_pred - nominal_physical_pred])))))
    axes[3].plot([-lim, lim], [-lim, lim], color="black", linewidth=1)
    axes[3].axhline(0, color="gray", linewidth=0.8)
    axes[3].axvline(0, color="gray", linewidth=0.8)
    axes[3].set_xlim(-lim, lim)
    axes[3].set_ylim(-lim, lim)
    axes[3].set_xlabel("hidden physical true - nominal physical")
    axes[3].set_ylabel("calibrated physical - nominal physical")
    axes[3].set_title("did calibration learn the mismatch?")
    axes[3].grid(True, alpha=0.3)

    plt.show()

    print("measured MAE nominal -> data:", float(nominal_measured_mae))
    print("measured MAE calibrated -> data:", float(calibrated_measured_mae))
    print("hidden true MAE nominal -> true:", float(nominal_true_mae))
    print("hidden true MAE calibrated -> true:", float(calibrated_true_mae))
else:
    print("No dataset yet.")



## Export current best pulse


In [ ]:

print("best_controls shape:", best_controls.shape)
print("max abs coefficient:", float(jnp.max(jnp.abs(best_controls))))
print("best measured P_n:", best_measured)
print("best lower confidence bound:", best_lcb)
print("best true diagnostic P_n:", best_true_diagnostic)
